In [18]:
import pandas as pd
import os

NEW_MODEL_INPUT = r'C:\projects\IdahoSTDM\ITD_STDM\IdahoSTDM\ITDSTDM\inputs2020'
NEW_MODEL_PATH = r'C:\projects\IdahoSTDM\ITD_STDM\IdahoSTDM\ITDSTDM\outputs2020_newTAZ'

new_taz = pd.read_csv(os.path.join(NEW_MODEL_INPUT, 'tazs.csv'))

col_names = ['home_state_fips', 'home_county_fips', 'home_state_name', 'home_county_name', 
             'work_state_fips', 'work_county_fips', 'work_state_name', 'work_county_name',
             'commute_flow_estimate', 'commute_flow_moe']

jtw_data = pd.read_excel(r'C:\projects\IdahoSTDM\Census_data\work_county_flows_acs5_2020.xlsx', skiprows = 8, names = col_names)
jtw_data = jtw_data[~jtw_data['work_state_fips'].isna()]

persons = pd.read_csv(os.path.join(NEW_MODEL_PATH, 'persondata.csv')).merge(
    new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'home_state', 'County': 'home_county_name'}),
    how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ'
    )
workers = persons[persons['WORK_TAZ'] > 0].merge(
    new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'work_state', 'County': 'work_county_name'}),
    how = 'left', left_on = 'WORK_TAZ', right_on = 'STDM_TAZ'
    )

workers['home_county_name'] = workers['home_county_name'] + ' County'
workers['work_county_name'] = workers['work_county_name'] + ' County'


In [31]:
flow_compare = workers[(workers['home_state'] == 'Idaho') & (workers['work_state'] == 'Idaho')].groupby(['home_county_name', 'work_county_name']).agg(model = ('memberID', 'count')).join(
    jtw_data[(jtw_data['home_state_name'] == 'Idaho') & (jtw_data['work_state_name'] == 'Idaho')][['home_county_name', 'work_county_name', 'commute_flow_estimate']].set_index(['home_county_name', 'work_county_name'])
).fillna(0)

flow_compare['difference'] = flow_compare['commute_flow_estimate'] - flow_compare['model']

with pd.ExcelWriter('cf_crosstab.xlsx') as writer:  
    flow_compare.reset_index().pivot(index = 'home_county_name', columns = 'work_county_name', values = 'commute_flow_estimate').fillna(0).to_excel(writer, sheet_name='acs')
    flow_compare.reset_index().pivot(index = 'home_county_name', columns = 'work_county_name', values = 'model').fillna(0).to_excel(writer, sheet_name='model')
    flow_compare.reset_index().pivot(index = 'home_county_name', columns = 'work_county_name', values = 'difference').fillna(0).to_excel(writer, sheet_name='diff')



In [13]:
jtw_data[(jtw_data['home_state_name'] == 'Idaho') & (jtw_data['work_state_name'] == 'Idaho')]

,home_state_fips,home_county_fips,home_state_name,home_county_name,work_state_fips,work_county_fips,work_state_name,work_county_name,commute_flow_estimate,commute_flow_moe
22857,16,1.0,Idaho,Ada County,16.0,1.0,Idaho,Ada County,220996.0,2495.0
22858,16,1.0,Idaho,Ada County,16.0,3.0,Idaho,Adams County,3.0,8.0
22859,16,1.0,Idaho,Ada County,16.0,5.0,Idaho,Bannock County,15.0,24.0
22860,16,1.0,Idaho,Ada County,16.0,9.0,Idaho,Benewah County,69.0,101.0
22861,16,1.0,Idaho,Ada County,16.0,13.0,Idaho,Blaine County,102.0,77.0
...,...,...,...,...,...,...,...,...,...,...
23767,16,87.0,Idaho,Washington County,16.0,3.0,Idaho,Adams County,17.0,17.0
23768,16,87.0,Idaho,Washington County,16.0,27.0,Idaho,Canyon County,80.0,57.0
23769,16,87.0,Idaho,Washington County,16.0,45.0,Idaho,Gem County,14.0,21.0
23770,16,87.0,Idaho,Washington County,16.0,75.0,Idaho,Payette County,125.0,67.0


model
home_county       work_county              
Ada County        Ada County         260250
                  Adams County            1
                  Bannock County         67
                  Bingham County          8
                  Blaine County          54
...                                     ...
Washington County Owyhee County          13
                  Payette County        104
                  Twin Falls County      11
                  Valley County           6
                  Washington County    3507

[785 rows x 1 columns]